# 05 · Integrated Analytics – MongoDB + HBase + Spark working together
**Part 3 of the assignment (25 %).** Two integrated analytical queries (the assignment asks for one – we do two):

| | Query A | Query B |
|---|---|---|
| **Business question** | *Which customers will be worth the most over the next 12 months, and do engagement signals help predict it?* | *Where do shopping sessions drop out between first product view and purchase, and how much revenue is lost there?* |
| **Technique** | Customer Lifetime Value + predictive validation + segmentation (KMeans) | Funnel conversion analysis with data-quality reconciliation |
| **MongoDB provides** | user profile + embedded purchase summary, transaction history | the true list of purchases (`transactions.session_id`) |
| **HBase provides** | per-user engagement (sessions, minutes, page views) via a `meta`-family scan | the session stream (stages: product view → cart → checkout) |
| **Spark does** | cross-system join, feature engineering, MLlib models | stage flags, conditional aggregation, breakdowns |

## Architecture – where each system earns its place
```
                    ┌─────────────────────────┐              ┌──────────────────────────┐
  profiles,         │  MongoDB                │              │  HBase                   │   session stream,
  orders,   ───────►│  flexible documents,    │              │  user#reverse_ts rows,   │◄─────── page views,
  catalogue         │  rich aggregations      │              │  fast per-user key scans │         cart lines
                    └───────────┬─────────────┘              └─────────────┬────────────┘
                                │  pymongo / connector                     │  happybase / connector
                                ▼                                          ▼
                    ┌───────────────────────────────────────────────────────────────────┐
                    │                        Apache Spark                                │
                    │   joins ACROSS systems · window functions · MLlib · large scans    │
                    └──────────────────────────────┬────────────────────────────────────┘
                                                   ▼
                                  result tables (CSV/Parquet) ─► notebook 06 charts
```
**Division of labour (the key idea for the report):** each store answers the question it is *built* for (Mongo: flexible analytics on documents; HBase: fast keyed/time-series access). **Spark is the only place where the two meet**, because neither store can join with the other.

> **Live vs simulated:** the loaders print which source they used. If Docker (Mongo/HBase) is running the data comes from the stores; otherwise from the identical Parquet files (`SIMULATED`). Report which one you ran.

In [1]:
import sys; sys.path.append("../src")
from config import *
import datetime as dt
import pandas as pd, numpy as np
from pyspark.sql import functions as F, Window
import federation as FED

spark = get_spark("05-integrated")
PQ = PROCESSED / "parquet"
rd = lambda n: spark.read.parquet(str(PQ / n))
sessions, txns, items, users = rd("sessions"), rd("transactions").filter("is_valid"), rd("transaction_items"), rd("users")
pd.set_option("display.max_columns", 40, "display.width", 220)
print({"sessions": sessions.count(), "orders": txns.count(), "users": users.count()})

{'sessions': 200000, 'orders': 50000, 'users': 10000}


---
# QUERY A · Customer Lifetime Value (CLV)

### A.1 Extract – one DataFrame from each system
* **MongoDB →** `mongo_users`: profile + the embedded `purchase_summary` (orders, total spent, AOV, first/last purchase).
* **HBase →** `hbase_eng`: per-user engagement aggregated from a scan of the small **`meta`** column family only (the heavy page-view family is never read).

In [2]:
with timer("extract MongoDB users"):   mongo_users, mongo_label = FED.mongo_users_df(spark)
with timer("extract HBase engagement"): hbase_eng, hbase_label = FED.hbase_engagement_df(spark)
mongo_users, hbase_eng = mongo_users.cache(), hbase_eng.cache()
print("MongoDB side:", mongo_label, "|", mongo_users.count(), "users")
print("HBase side  :", hbase_label, "|", hbase_eng.count(), "users with sessions")

[timer] extract MongoDB users: 0.89s
[timer] extract HBase engagement: 9.63s
MongoDB side: MongoDB (LIVE: users collection) | 10000 users
HBase side  : HBase (LIVE: user_sessions scan, meta family) | 10000 users with sessions


### A.2 Transform – join across systems and add margin
Gross profit comes from the transaction **line items** × the **category margin** (see Limitations: margins exist only per category in this data). Missing engagement (user with no sessions) becomes 0.

In [3]:
gp = items.groupBy("user_id").agg(F.round(F.sum("est_gross_profit"), 2).alias("gross_profit"),
                                  F.countDistinct("category_id").alias("n_categories"),
                                  F.round(F.sum("item_net_revenue"), 2).alias("net_revenue"))
disc = txns.groupBy("user_id").agg(F.round(F.avg("discount_rate"), 3).alias("avg_discount_rate"),
                                   F.round(F.avg(F.col("has_session").cast("int")), 3).alias("share_orders_with_session"))

base = (mongo_users.join(hbase_eng, "user_id", "left").join(gp, "user_id", "left").join(disc, "user_id", "left")
        .fillna({"sessions": 0, "total_seconds": 0, "total_page_views": 0, "gross_profit": 0.0, "n_categories": 0,
                 "net_revenue": 0.0, "avg_discount_rate": 0.0, "share_orders_with_session": 0.0})
        .withColumn("minutes_browsing", F.round(F.col("total_seconds") / 60, 1)))

# Observation length in months (the data cover ~90 days)
d0, d1 = sessions.agg(F.min("session_date"), F.max("session_date")).first()
obs_months = ((d1 - d0).days + 1) / 30
print(f"Observation window: {d0} -> {d1} = {obs_months:.2f} months")

Observation window: 2026-08-11 -> 2026-11-09 = 3.03 months


### A.3 Descriptive CLV (12-month, profit-based)
With only 90 days of history, **churn cannot be observed**, so we make the assumption explicit and keep it as visible parameters:

$$\text{CLV}_{12m} \;=\; \underbrace{\frac{\text{gross profit observed}}{\text{months observed}}}_{\text{monthly margin } m}\;\times\;\sum_{t=0}^{11}\frac{r^{t}}{(1+d)^{t}}$$

* **r** = monthly retention (probability the customer is still active next month) – *assumption*
* **d** = monthly discount rate – *assumption*

Change `RETENTION` / `DISCOUNT` below and re-run to show sensitivity in the report.

In [4]:
RETENTION, DISCOUNT = 0.90, 0.01                          # <- assumptions (sensitivity analysis: try 0.80 / 0.95)
horizon_factor = sum((RETENTION ** t) / ((1 + DISCOUNT) ** t) for t in range(12))
print(f"12-month retention/discount multiplier = {horizon_factor:.3f} (a flat 12.000 would ignore churn and time value)")

clv = (base.withColumn("monthly_margin", F.round(F.col("gross_profit") / F.lit(obs_months), 2))
           .withColumn("clv_12m", F.round(F.col("monthly_margin") * F.lit(horizon_factor), 2)))
clv_pd = clv.toPandas().sort_values("clv_12m", ascending=False)
display(clv_pd[["user_id", "country", "orders", "total_spent", "gross_profit", "sessions", "minutes_browsing", "clv_12m"]].head(10))

dec = clv_pd.assign(decile=pd.qcut(clv_pd.clv_12m.rank(method="first"), 10, labels=range(1, 11))).groupby("decile", observed=True).agg(
    users=("user_id", "count"), avg_clv=("clv_12m", "mean"), total_clv=("clv_12m", "sum")).round(0)
dec["share_of_total_pct"] = (dec.total_clv / dec.total_clv.sum() * 100).round(1)
dec.to_csv(TAB / "int_clv_deciles.csv"); display(dec.sort_index(ascending=False))
top10_share = dec.loc[10, "share_of_total_pct"]
print(f"Top 10% of customers = {top10_share}% of projected 12-month profit  |  total projected profit: {clv_pd.clv_12m.sum():,.0f}")

12-month retention/discount multiplier = 6.880 (a flat 12.000 would ignore churn and time value)


,user_id,country,orders,total_spent,gross_profit,sessions,minutes_browsing,clv_12m
3258,user_003258,RU,16,20441.43,5305.35,24,668.0,12034.08
7368,user_007368,SB,11,19230.90,4952.86,18,480.0,11234.50
9161,user_009161,AG,11,18453.05,4825.02,16,502.2,10944.56
4921,user_004921,CG,13,18406.05,4755.26,17,487.4,10786.31
9189,user_009189,BF,12,18031.18,4496.95,19,636.7,10200.37
883,user_000883,VA,12,17247.24,4443.84,22,723.4,10079.89
1153,user_001153,BB,11,16637.22,4394.33,20,648.6,9967.60
5480,user_005480,CV,9,16696.80,4355.21,25,755.8,9878.84
7764,user_007764,BY,14,16851.54,4287.32,26,993.3,9724.86
1490,user_001490,GQ,14,16762.99,4263.70,21,524.2,9671.33


,users,avg_clv,total_clv,share_of_total_pct
decile,,,,
10,1000,6358.0,6358272.0,20.6
9,1000,4808.0,4807572.0,15.6
8,1000,4085.0,4084510.0,13.2
7,1000,3559.0,3559114.0,11.5
6,1000,3123.0,3122910.0,10.1
5,1000,2730.0,2729945.0,8.8
4,1000,2326.0,2326021.0,7.5
3,1000,1880.0,1880200.0,6.1
2,1000,1381.0,1380557.0,4.5


Top 10% of customers = 20.6% of projected 12-month profit  |  total projected profit: 30,888,725


### A.4 Do engagement signals actually predict future spend? *(time-split validation with Spark MLlib)*
A CLV formula is only useful if it points at real behaviour. We test it honestly with a **time split** (no leakage):

* **Features** – everything known in the **first 60 days**: orders, spend, sessions, minutes, page views, carts, checkouts, recency.
* **Target** – spend in the **last 30 days** (never seen by the features).
* **Baseline** – "spend continues at the same run-rate" (`spend_60 / 2`). A model is only worth something if it **beats the baseline** on held-out users (80/20 split).

> *Architecture note:* the "sessions in the first 60 days" feature is a **global time-window** query. HBase's row key is `user#time`, so that access pattern is **not** served by the key (it would be a full scan with a filter) – this is precisely the work that belongs in Spark. A trade-off of the key design worth stating in the report.

In [5]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

cut = d0 + dt.timedelta(days=60)
hist  = txns.filter(F.col("txn_date") < F.lit(cut)).groupBy("user_id").agg(
          F.count("*").alias("orders_60"), F.sum("total").alias("spend_60"), F.max("txn_date").alias("last_order"))
fut   = txns.filter(F.col("txn_date") >= F.lit(cut)).groupBy("user_id").agg(F.sum("total").alias("spend_next30"))
sess60 = sessions.filter(F.col("session_date") < F.lit(cut)).groupBy("user_id").agg(
          F.count("*").alias("sessions_60"), F.sum("duration_seconds").alias("seconds_60"), F.sum("n_page_views").alias("views_60"),
          F.sum(F.col("has_cart").cast("int")).alias("carts_60"), F.sum(F.col("reached_checkout").cast("int")).alias("checkouts_60"))

data = (users.select("user_id").join(hist, "user_id", "left").join(sess60, "user_id", "left").join(fut, "user_id", "left")
        .withColumn("recency_days_60", F.coalesce(F.datediff(F.lit(cut), F.col("last_order")), F.lit(60)))
        .drop("last_order").fillna(0))
FEATURES = ["orders_60", "spend_60", "sessions_60", "seconds_60", "views_60", "carts_60", "checkouts_60", "recency_days_60"]
asm = VectorAssembler(inputCols=FEATURES, outputCol="features")
train, test = data.randomSplit([0.8, 0.2], seed=42)
print(f"features from < {cut} | target = spend in [{cut}, {d1}] | train {train.count():,} / test {test.count():,} users")

ev = {m: RegressionEvaluator(labelCol="spend_next30", predictionCol="prediction", metricName=m) for m in ("rmse", "mae", "r2")}
rows = []
# baseline: run-rate
base_pred = test.withColumn("prediction", F.col("spend_60") / 2)
rows.append({"model": "Baseline (run-rate: spend_60 / 2)", **{m: round(e.evaluate(base_pred), 3) for m, e in ev.items()}})
# models
lr = Pipeline(stages=[asm, LinearRegression(featuresCol="features", labelCol="spend_next30", standardization=True)]).fit(train)
rf = Pipeline(stages=[asm, RandomForestRegressor(featuresCol="features", labelCol="spend_next30", numTrees=100, maxDepth=5, seed=42)]).fit(train)
for name, mdl in [("Linear regression", lr), ("Random forest (100 trees)", rf)]:
    pred = mdl.transform(test)
    rows.append({"model": name, **{m: round(e.evaluate(pred), 3) for m, e in ev.items()}})
metrics = pd.DataFrame(rows); metrics.to_csv(TAB / "int_model_metrics.csv", index=False); display(metrics)

imp = pd.Series(rf.stages[-1].featureImportances.toArray(), index=FEATURES).sort_values(ascending=False).round(3)
print("Random-forest feature importance:"); display(imp.to_frame("importance").T)

features from < 2026-10-10 | target = spend in [2026-10-10, 2026-11-09] | train 8,079 / test 1,921 users


,model,rmse,mae,r2
0,Baseline (run-rate: spend_60 / 2),2053.770,1608.396,-0.544
1,Linear regression,1654.522,1317.069,-0.002
2,Random forest (100 trees),1656.008,1317.151,-0.004


Random-forest feature importance:


,spend_60,views_60,carts_60,sessions_60,recency_days_60,seconds_60,checkouts_60,orders_60
importance,0.152,0.144,0.129,0.125,0.121,0.121,0.118,0.09


> **How to read this table:** compare **RMSE** (lower is better) and **R²** (higher is better; 0 = no better than predicting the average) with the baseline.
> * If the models **beat** the baseline, engagement carries signal beyond past spend – quote the gain.
> * If they **do not**, say so: the generator creates sessions and orders mostly independently, so *little* predictive structure is expected. Reporting a negative or near-zero result honestly, with the reason, is stronger than a forced positive story.

### A.5 Customer segmentation (KMeans on value + engagement)
Features are **standardised** (KMeans uses Euclidean distance, so a feature in dollars would swamp one in sessions). `k` is chosen by the **silhouette score** over k = 2…6.

In [6]:
from pyspark.ml.feature import StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

SEG = ["orders", "total_spent", "avg_order_value", "sessions", "minutes_browsing", "gross_profit"]
sa = VectorAssembler(inputCols=SEG, outputCol="raw")
seg_df = sa.transform(base.filter("orders > 0"))                       # segment customers (buyers) only
scaled = StandardScaler(inputCol="raw", outputCol="features", withMean=True, withStd=True).fit(seg_df).transform(seg_df).cache()

sil = {}
for k in range(2, 7):
    m = KMeans(k=k, seed=42, featuresCol="features").fit(scaled)
    sil[k] = ClusteringEvaluator(featuresCol="features").evaluate(m.transform(scaled))
display(pd.Series(sil, name="silhouette").round(3).to_frame().T)
best_k = max(sil, key=sil.get); print("Chosen k =", best_k, "(highest silhouette)")

,2,3,4,5,6
silhouette,0.456,0.391,0.384,0.377,0.345


Chosen k = 2 (highest silhouette)


In [7]:
model = KMeans(k=best_k, seed=42, featuresCol="features").fit(scaled)
segs = model.transform(scaled).select("user_id", F.col("prediction").alias("cluster")).toPandas()
full = clv_pd.merge(segs, on="user_id", how="left")

prof = full.dropna(subset=["cluster"]).groupby("cluster").agg(
    users=("user_id", "count"), avg_orders=("orders", "mean"), avg_spend=("total_spent", "mean"), avg_order_value=("avg_order_value", "mean"),
    avg_sessions=("sessions", "mean"), avg_minutes=("minutes_browsing", "mean"), avg_clv_12m=("clv_12m", "mean")).round(1)

# Give each cluster a business name from its z-scores (value axis vs engagement axis)
z = (prof - prof.mean()) / prof.std(ddof=0).replace(0, 1)
def label(r):
    v, e = r["avg_spend"], r["avg_sessions"]
    if v > .5 and e > .5:   return "Engaged big spenders"
    if v > .5:              return "Big spenders, light browsing"
    if v < -.5 and e > .5:  return "Browsers (engaged, low spend)"
    if v < -.5:             return "Low value / low engagement"
    return "Core regulars"
prof["segment"] = z.apply(label, axis=1)
prof["share_of_users_pct"] = (prof.users / prof.users.sum() * 100).round(1)
prof["share_of_clv_pct"] = (prof.avg_clv_12m * prof.users / (prof.avg_clv_12m * prof.users).sum() * 100).round(1)
full["segment"] = full["cluster"].map(prof["segment"]).fillna("Non-buyer")
display(prof.sort_values("avg_clv_12m", ascending=False))
prof.reset_index().to_csv(TAB / "int_segment_profile.csv", index=False)
full.to_csv(TAB / "int_user_clv.csv", index=False)
print("Saved int_user_clv.csv (", len(full), "users ) and int_segment_profile.csv")

,users,avg_orders,avg_spend,avg_order_value,avg_sessions,avg_minutes,avg_clv_12m,segment,share_of_users_pct,share_of_clv_pct
cluster,,,,,,,,,,
1.0,4096,6.9,8041.3,1204.7,21.7,661.3,4611.0,Engaged big spenders,41.2,61.1
0.0,5846,3.7,3590.5,987.6,18.8,565.8,2053.1,Low value / low engagement,58.8,38.9


Saved int_user_clv.csv ( 10000 users ) and int_segment_profile.csv


**Workflow summary for Query A (copy into the report)**
1. **MongoDB** → `users` documents with embedded `purchase_summary` (one collection read, no join over transactions).
2. **HBase** → one scan of the `meta` family, aggregated per user (column-family pruning keeps the read light).
3. **Spark** → joins the two with line-item gross profit, computes CLV, trains MLlib models on a time-split, clusters customers.
4. **Why this split?** The profile + purchase summary is a *document-shaped, read-by-key* need (Mongo); engagement is a *high-volume event stream* (HBase); the join and the ML are *scan-heavy and multi-source* (Spark).
5. **Performance:** the extract steps print their timings above; the HBase scan is the slowest step (single-threaded Thrift client). In production the HBase-Spark connector would parallelise it by region.

---
# QUERY B · Funnel Conversion Analysis

### B.1 Business question
*"Of all sessions, how many view a product, add to cart, reach checkout and really buy – where is the biggest drop, does it differ by device or referrer, and how much money is left on the table?"*

**Sources:** session behaviour (HBase `pv` + `cart` families → here the same data in Spark/Parquet), **real purchases from MongoDB's `transactions.session_id`**.

### B.2 First, a consistency check between the two systems
The generator's `conversion_status` says "converted" for sessions whose order was never created (stock failures). We therefore define a purchase as *"a transaction exists for the session"* – and, if MongoDB is running, **verify the Spark-derived set against MongoDB itself**.

In [8]:
purchased_spark = {r[0] for r in txns.filter("has_session").select("session_id").distinct().collect()}
if mongo_available():
    from pymongo import MongoClient
    purchased_mongo = set(x for x in MongoClient(MONGO_URI)[MONGO_DB].transactions.distinct("session_id") if x)
    print(f"Spark-derived purchased sessions: {len(purchased_spark):,} | MongoDB distinct session_id: {len(purchased_mongo):,} | identical: {purchased_spark == purchased_mongo}")
else:
    print(f"MongoDB not running - using the Spark/Parquet transaction set only ({len(purchased_spark):,} purchased sessions).")

Spark-derived purchased sessions: 25,094 | MongoDB distinct session_id: 25,094 | identical: True


### B.3 Strict funnel (each stage requires the previous one)
| Stage | Definition |
|---|---|
| 0 Sessions | all sessions |
| 1 Viewed a product | at least one `product_detail` page view |
| 2 Added to cart | viewed a product **and** cart non-empty |
| 3 Reached checkout | … **and** visited the `checkout` page |
| 4 Purchased | … **and** a real transaction exists |

In [9]:
fl = (sessions.select("session_id", "device_type", "referrer", "conversion_status_raw", "cart_value", "duration_seconds", "has_cart",
                      "viewed_product_page", "reached_checkout", "is_purchased")
      .withColumn("s1", F.col("viewed_product_page").cast("int"))
      .withColumn("s2", (F.col("viewed_product_page") & F.col("has_cart")).cast("int"))
      .withColumn("s3", (F.col("viewed_product_page") & F.col("has_cart") & F.col("reached_checkout")).cast("int"))
      .withColumn("s4", (F.col("viewed_product_page") & F.col("has_cart") & F.col("reached_checkout") & F.col("is_purchased")).cast("int")).cache())

outside = fl.filter(F.col("is_purchased") & (F.col("s4") == 0)).count()
print("Purchases that fall outside the strict funnel (should be 0):", outside)

STAGES = [("s0_sessions", "Sessions"), ("s1", "Viewed product"), ("s2", "Added to cart"), ("s3", "Reached checkout"), ("s4", "Purchased")]
def funnel(by=None):
    aggs = [F.count("*").alias("s0_sessions")] + [F.sum(c).alias(c) for c, _ in STAGES[1:]]
    g = (fl.groupBy(by) if by else fl.groupBy()).agg(*aggs).toPandas()
    return g

overall = funnel().iloc[0]
tbl = pd.DataFrame({"stage": [n for _, n in STAGES], "sessions": [int(overall[c]) for c, _ in STAGES]})
tbl["pct_of_all_sessions"] = (tbl.sessions / tbl.sessions.iloc[0] * 100).round(1)
tbl["step_conversion_pct"] = (tbl.sessions / tbl.sessions.shift(1) * 100).round(1)
tbl["drop_off_sessions"] = (tbl.sessions.shift(1) - tbl.sessions).fillna(0).astype(int)
tbl.to_csv(TAB / "int_funnel_overall.csv", index=False); display(tbl)
biggest = tbl.iloc[1:].sort_values("drop_off_sessions", ascending=False).iloc[0]
print(f"Largest absolute drop-off: before '{biggest.stage}' ({int(biggest.drop_off_sessions):,} sessions lost)")

Purchases that fall outside the strict funnel (should be 0): 0


,stage,sessions,pct_of_all_sessions,step_conversion_pct,drop_off_sessions
0,Sessions,200000,100.0,NaN,0
1,Viewed product,197422,98.7,98.7,2578
2,Added to cart,130170,65.1,65.9,67252
3,Reached checkout,58036,29.0,44.6,72134
4,Purchased,25094,12.5,43.2,32942


Largest absolute drop-off: before 'Reached checkout' (72,134 sessions lost)


### B.4 Breakdown by device and by referrer

In [10]:
def breakdown(col):
    g = funnel(col).rename(columns={"s1": "viewed_product", "s2": "added_to_cart", "s3": "reached_checkout", "s4": "purchased"})
    g["view_to_cart_pct"] = (g.added_to_cart / g.viewed_product * 100).round(1)
    g["cart_to_checkout_pct"] = (g.reached_checkout / g.added_to_cart * 100).round(1)
    g["checkout_to_purchase_pct"] = (g.purchased / g.reached_checkout * 100).round(1)
    g["overall_purchase_pct"] = (g.purchased / g.s0_sessions * 100).round(2)
    return g.sort_values("overall_purchase_pct", ascending=False)

by_dev, by_ref = breakdown("device_type"), breakdown("referrer")
by_dev.to_csv(TAB / "int_funnel_by_device.csv", index=False); by_ref.to_csv(TAB / "int_funnel_by_referrer.csv", index=False)
display(by_dev); display(by_ref)
spread = by_ref.overall_purchase_pct.max() - by_ref.overall_purchase_pct.min()
print(f"Spread between best and worst referrer: {spread:.2f} percentage points  (with ~{int(by_ref.s0_sessions.min()):,}+ sessions per group, differences under ~1pp are within noise)")

,device_type,s0_sessions,viewed_product,added_to_cart,reached_checkout,purchased,view_to_cart_pct,cart_to_checkout_pct,checkout_to_purchase_pct,overall_purchase_pct
1,tablet,66594,65767,43400,19519,8390,66.0,45.0,43.0,12.60
2,desktop,66873,66014,43458,19285,8416,65.8,44.4,43.6,12.59
0,mobile,66533,65641,43312,19232,8288,66.0,44.4,43.1,12.46


,referrer,s0_sessions,viewed_product,added_to_cart,reached_checkout,purchased,view_to_cart_pct,cart_to_checkout_pct,checkout_to_purchase_pct,overall_purchase_pct
2,search_engine,39646,39095,25893,11713,5080,66.2,45.2,43.4,12.81
0,direct,40236,39726,26182,11696,5135,65.9,44.7,43.9,12.76
4,affiliate,39897,39371,25891,11558,5058,65.8,44.6,43.8,12.68
3,social,40116,39634,26130,11665,4970,65.9,44.6,42.6,12.39
1,email,40105,39596,26074,11404,4851,65.9,43.7,42.5,12.10


Spread between best and worst referrer: 0.71 percentage points  (with ~39,646+ sessions per group, differences under ~1pp are within noise)


### B.5 How much money is at stake?
Three numbers for the business: value of carts that were **purchased**, **abandoned** (never reached an order), and – the data-quality finding – carts of sessions that *looked* converted but have **no order** (typically a stock-out at checkout = *lost* revenue, not abandonment).

In [11]:
v = fl.agg(
    F.sum(F.when(F.col("is_purchased"), F.col("cart_value")).otherwise(0)).alias("purchased_cart_value"),
    F.sum(F.when(F.col("has_cart") & ~F.col("is_purchased") & (F.col("conversion_status_raw") != "converted"), F.col("cart_value")).otherwise(0)).alias("abandoned_cart_value"),
    F.sum(F.when((F.col("conversion_status_raw") == "converted") & ~F.col("is_purchased"), F.col("cart_value")).otherwise(0)).alias("failed_checkout_value"),
    F.sum(F.when((F.col("conversion_status_raw") == "converted") & ~F.col("is_purchased"), 1).otherwise(0)).alias("failed_checkout_sessions"),
    F.sum(F.when(F.col("conversion_status_raw") == "converted", 1).otherwise(0)).alias("raw_converted_sessions"),
    F.sum(F.col("is_purchased").cast("int")).alias("real_purchases")).toPandas().iloc[0]
value = pd.DataFrame({"metric": ["Cart value of purchased sessions", "Cart value abandoned (never 'converted')", "Cart value of failed checkouts (looked converted, no order)",
                                 "Sessions flagged 'converted' by the platform", "Sessions with a real order", "Platform over-reports conversions by"],
                      "value": [round(v.purchased_cart_value), round(v.abandoned_cart_value), round(v.failed_checkout_value),
                                int(v.raw_converted_sessions), int(v.real_purchases),
                                f"{(v.raw_converted_sessions / v.real_purchases - 1) * 100:.0f}%"]})
value.to_csv(TAB / "int_funnel_value.csv", index=False); display(value)

,metric,value
0,Cart value of purchased sessions,19986282
1,Cart value abandoned (never 'converted'),71950644
2,Cart value of failed checkouts (looked convert...,12312826
3,Sessions flagged 'converted' by the platform,40609
4,Sessions with a real order,25094
5,Platform over-reports conversions by,62%


In [12]:
# Do purchasers behave differently? (supports the "what to improve" recommendation)
display(fl.groupBy("conversion_status_raw", "is_purchased").agg(
    F.count("*").alias("sessions"), F.round(F.avg("duration_seconds") / 60, 1).alias("avg_minutes"), F.round(F.avg("cart_value"), 0).alias("avg_cart_value"))
    .orderBy("conversion_status_raw", "is_purchased").toPandas())

,conversion_status_raw,is_purchased,sessions,avg_minutes,avg_cart_value
0,abandoned,False,89561,30.2,803.0
1,browsed,False,69830,30.2,0.0
2,converted,False,15515,30.4,794.0
3,converted,True,25094,30.2,796.0


**Workflow summary for Query B (copy into the report)**
1. **HBase side:** a per-user prefix scan serves *"show this customer's journey"* in milliseconds (notebook 02). The *funnel across all users* is a full-scan aggregation – so it runs in **Spark** on the same session data (`pv`/`cart` families ⇄ `page_views`/`cart_items` Parquet).
2. **MongoDB side:** the `transactions` collection is the system of record for purchases; the indexed `session_id` lets us verify (B.2) which sessions truly bought.
3. **Spark:** stage flags with `exists()` on arrays, conditional aggregation in one pass, breakdowns by device/referrer.
4. **Performance:** one shuffle-light pass over the sessions table, cached once (`fl.cache()`) because five queries reuse it.

In [13]:
spark.stop()

### ✅ What you should have after this notebook
`outputs/tables/`: `int_user_clv`, `int_clv_deciles`, `int_model_metrics`, `int_segment_profile`, `int_funnel_overall`, `int_funnel_by_device`, `int_funnel_by_referrer`, `int_funnel_value`.

➡ Next: `06_visualization_and_insights.ipynb`.